# DLinear: tái lập SGD và kiểm dự đoán 1.1

Thứ tự cell đã được sắp lại để **Restart Kernel → Run All** chạy được từ trên xuống, không phụ thuộc biến cũ trong kernel.
Mọi hàm lấy từ `src/` (mô hình ở `src/models.py`, huấn luyện ở `src/sgd.py`, dạng đóng ở `src/solvers.py`, `src/operators.py`).

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader

from src.data import load_ett, load_etth1, make_dataset
from src.metrics import sse
from src.models import DLinear, moving_avg
from src.operators import build_N, build_P, make_Z, nlinear_transform
from src.sgd import ETTDataset, evaluate, train_sgd
from src.solvers import fit_with_bias, lstsq_solver, nlinear_constrained, ols, ridge, ridge_pen

## 1. Mô hình DLinear

Định nghĩa ở `src/models.py` (giống `models/DLinear.py` của LTSF-Linear với `individual=False`).

In [2]:
model = DLinear(seq_len=336, pred_len=96)
print(model(torch.randn(32, 336, 7)).shape)   # kỳ vọng: [32, 96, 7]

torch.Size([32, 96, 7])


## 2. Dữ liệu cho baseline SGD

In [3]:
path = '../data/ETTh1.csv'
df = pd.read_csv(path)
data = df.iloc[:, 1:].values
print(df.shape, data.shape)     # kỳ vọng: (17420, 8) (17420, 7)

(17420, 8) (17420, 7)


In [4]:
seq_len, pred_len = 336, 96
train_data, val_data, test_data = load_ett(path, seq_len, "ETTh1")   # chuẩn hoá theo train, ddof=0 như StandardScaler

train_ds = ETTDataset(train_data, seq_len, pred_len)
val_ds   = ETTDataset(val_data,   seq_len, pred_len)
test_ds  = ETTDataset(test_data,  seq_len, pred_len)
print(len(train_ds), len(val_ds), len(test_ds))   # kỳ vọng: 8209 2785 2785

x, y = train_ds[0]
print(x.shape, y.shape)                           # kỳ vọng: [336, 7] [96, 7]

8209 2785 2785
torch.Size([336, 7]) torch.Size([96, 7])


In [5]:
torch.manual_seed(42)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

## 3. Huấn luyện DLinear bằng SGD

`train_sgd` (Adam, lr 0,005, dừng sớm patience 3) dùng lịch `lradj='type1'` của LTSF-Linear: epoch 1 và 2 cùng chạy lr = 0,005, rồi giảm một nửa mỗi epoch. Lần chạy trước dùng `StepLR`, giảm ngay sau epoch 1; checkpoint của lần đó giữ ở `dlinear_sgd_ETTh1_L336_H96.pt` (các test trong `tests/` dựa vào nó), lần chạy theo `type1` lưu ra file riêng.

In [6]:
CKPT_STEP  = "../checkpoints/dlinear_sgd_ETTh1_L336_H96.pt"          # StepLR, lần chạy trước
CKPT_TYPE1 = "../checkpoints/dlinear_sgd_ETTh1_L336_H96_type1.pt"    # lradj='type1'

torch.manual_seed(42)
model = DLinear(336, 96)
history = train_sgd(model, train_loader, val_loader, CKPT_TYPE1, lr=5e-3, epochs=10, patience=3, lradj="type1")
for epoch, train_loss, val_loss, lr_next in history:
    print(f"epoch {epoch}: train {train_loss:.4f} | val {val_loss:.4f} | lr epoch sau {lr_next:.6f}")

epoch 1: train 0.4029 | val 0.7205 | lr epoch sau 0.005000
epoch 2: train 0.3813 | val 0.7094 | lr epoch sau 0.002500
epoch 3: train 0.3540 | val 0.6797 | lr epoch sau 0.001250
epoch 4: train 0.3427 | val 0.6536 | lr epoch sau 0.000625
epoch 5: train 0.3369 | val 0.6572 | lr epoch sau 0.000313
epoch 6: train 0.3340 | val 0.6538 | lr epoch sau 0.000156
epoch 7: train 0.3320 | val 0.6630 | lr epoch sau 0.000078


In [7]:
model_step = DLinear(336, 96)
model_step.load_state_dict(torch.load(CKPT_STEP))
for name, m in [("StepLR (checkpoint cũ)", model_step), ("type1 (LTSF-Linear)", model)]:
    print(f"{name:24s} Test MSE: {evaluate(m, test_loader, 'mse'):.4f} | MAE: {evaluate(m, test_loader, 'mae'):.4f}")

StepLR (checkpoint cũ)   Test MSE: 0.3766 | MAE: 0.3996


type1 (LTSF-Linear)      Test MSE: 0.3825 | MAE: 0.4048


Lần chạy 1: Test MSE 0.3779 | MAE 0.3999, tính bằng trung bình theo batch (batch cuối chỉ có 1 mẫu nên bị lệch). `evaluate` của `src/sgd.py` lấy trung bình toàn cục trên mọi phần tử, cho checkpoint StepLR 0.3766 | 0.3996 (công bố 0.375 | 0.399). Đây là số dùng để so với OLS 0.3702.

## 4. Ma trận trung bình trượt `P`

In [8]:
torch.set_printoptions(precision=3)
print(torch.from_numpy(build_P(5, 5)) * 5)   # xem tận mắt: phải ra [3 1 1 0 0], [2 1 1 1 0], ...

tensor([[3.000, 1.000, 1.000, 0.000, 0.000],
        [2.000, 1.000, 1.000, 1.000, 0.000],
        [1.000, 1.000, 1.000, 1.000, 1.000],
        [0.000, 1.000, 1.000, 1.000, 2.000],
        [0.000, 0.000, 1.000, 1.000, 3.000]], dtype=torch.float64)


In [9]:
def probe_P(L: int, k: int) -> torch.Tensor:
    """Kênh i của đầu vào là e_i  =>  output kênh i là cột i của P."""
    ma = moving_avg(kernel_size=k, stride=1)
    E = torch.eye(L, dtype=torch.float64).unsqueeze(0)  # [1, L, L]
    return ma(E)[0]                                      # [L, L]

def check(L: int, k: int) -> None:
    P = torch.from_numpy(build_P(L, k))
    assert torch.allclose(P, probe_P(L, k)), f"P không khớp moving_avg (L={L}, k={k})"
    x = torch.randn(2, L, 3, dtype=torch.float64)
    trend = moving_avg(k, 1)(x)
    assert torch.allclose(trend, torch.einsum("ti,bic->btc", P, x))
    print(f"OK  L={L:3d}, k={k:2d}")

for L_, k_ in [(5, 3), (5, 5), (10, 7), (96, 25), (336, 25)]:
    check(L_, k_)

OK  L=  5, k= 3
OK  L=  5, k= 5
OK  L= 10, k= 7
OK  L= 96, k=25
OK  L=336, k=25


In [10]:
# Sai lệch lớn nhất giữa P @ x và moving_avg, cho các kernel size dùng ở 1.2
for k in [5, 15, 25, 49]:
    P_k = torch.from_numpy(build_P(336, k))
    x   = torch.randn(336, dtype=torch.float64)
    out = moving_avg(k, 1)(x.reshape(1, 336, 1)).reshape(-1)   # output của repo
    diff = (P_k @ x - out).abs().max()
    print(k, diff.item())

5 2.220446049250313e-16
15 1.1102230246251565e-16
25 2.220446049250313e-16
49 2.0816681711721685e-16


## 5. Công cụ dạng đóng

`load_etth1`, `make_dataset`, `ols`, `fit_with_bias`, `sse` lấy từ `src/` (đã import ở đầu notebook).

In [11]:
L, H = 336, 96
train, val, test = load_etth1(path, L)
X_train, Y_train = make_dataset(train, L, H)
X_test,  Y_test  = make_dataset(test,  L, H)
print(X_train.shape, X_test.shape)          # kỳ vọng: (57463, 336) (19495, 336)

W_ols, b_ols = fit_with_bias(ols, X_train, Y_train)
print(sse(W_ols, b_ols, X_test, Y_test) / Y_test.size)   # kỳ vọng: 0.3702..., khớp Nlinear.ipynb

(57463, 336) (19495, 336)


0.37023527067813916


## 6. Kiểm dự đoán 1.1: DLinear ≡ Linear khi λ = 0

Ý tưởng: dựng ma trận thiết kế của DLinear $Z = [\,XP^\top,\ X(I-P)^\top\,]$ kích thước $n \times 672$, giải bằng `lstsq` (chịu được ma trận suy biến), rồi so **dự báo** với Linear.

In [12]:
k = 25
P_np = build_P(L, k)
I = np.eye(L)

Z_train, Z_test = make_Z(X_train, P_np), make_Z(X_test, P_np)   # ghép ngang [X Pᵀ, X (I−P)ᵀ]
print(Z_train.shape)             # kỳ vọng: (57463, 672)

W_D, b_D = fit_with_bias(lstsq_solver, Z_train, Y_train)   # lstsq chịu được Z suy biến (hạng 336)

pred_D = Z_test @ W_D.T + b_D       # dự báo DLinear trên test
pred_L = X_test @ W_ols.T + b_ols   # dự báo Linear trên test
print(np.abs(pred_D - pred_L).max())

(57463, 672)


2.1271873151818e-13


In [13]:
for k in [5, 15, 25, 49]:
    # --- 1. Dựng N = PᵀP + (I−P)ᵀ(I−P) ---
    P_np = build_P(L, k)
    N_inv = np.linalg.inv(build_N(P_np))
    Z_train, Z_test = make_Z(X_train, P_np), make_Z(X_test, P_np)
    print(f"{k}:")

    # --- 2. Ba mô hình, hai mức lambda ---
    for lam in [1e3, 1e5]:
        # (a) DLinear có weight decay = ridge thường trên Z
        W_a, b_a = fit_with_bias(lambda X, Y: ridge(X, Y, lam), Z_train, Y_train)
        # (b) Linear ridge thường = ridge thường trên X
        W_b, b_b = fit_with_bias(lambda X, Y: ridge(X, Y, lam), X_train, Y_train)
        # (c) Linear ridge theo metric N = phạt lam * N_inv trên X
        W_c, b_c = fit_with_bias(lambda X, Y: ridge_pen(X, Y, lam * N_inv), X_train, Y_train)

        pred_a = Z_test @ W_a.T + b_a
        pred_b = X_test @ W_b.T + b_b
        pred_c = X_test @ W_c.T + b_c
        print(lam, np.abs(pred_a - pred_c).max(), np.abs(pred_a - pred_b).max())

5:


1000.0 1.9273471707492718e-13 0.12808761970499696


100000.0 1.2212453270876722e-14 0.18313537040674133


15:


1000.0 1.461053500406706e-13 0.19284763398561244


...

1000.0 1.723066134218243e-13 0.28165143678669313


100000.0 1.0880185641326534e-14 0.46140676773182654


49:


1000.0 1.5010215292932116e-13 0.3568760850268031


100000.0 1.021405182655144e-14 0.7617438122515077


In [14]:
Xt_train, Yt_train, _           = nlinear_transform(X_train, Y_train)
Xt_test,  Yt_test,  xlast_test  = nlinear_transform(X_test,  Y_test)

# --- Kiểm cầu nối ở λ = 0: cách nhìn mới phải trùng nghiệm có ràng buộc ở 1.3 ---
W_n, b_n = fit_with_bias(ols, Xt_train, Yt_train)
V, b_V = fit_with_bias(nlinear_constrained, X_train, Y_train)
pred_n = Xt_test @ W_n.T + b_n + xlast_test      # nhớ cộng lại x_L
pred_V = X_test @ V.T + b_V
print(np.abs(pred_n - pred_V).max())

# --- λ > 0: NLinear ridge so với Linear ridge ---
for lam in [1e3, 1e5]:
    W_n, b_n = fit_with_bias(lambda X, Y: ridge(X, Y, lam), Xt_train, Yt_train)
    pred_n = Xt_test @ W_n.T + b_n + xlast_test
    # Linear ridge: học trên dữ liệu gốc
    W_r, b_r = fit_with_bias(lambda X, Y: ridge(X, Y, lam), X_train, Y_train)
    pred_r = X_test @ W_r.T + b_r

    print(lam, np.abs(pred_n - pred_r).max())

3.956834859764058e-13


1000.0 0.4804574967269337


100000.0 2.8262209872555077


In [15]:
print(np.abs(xlast_test).max())

4.68069172001624
